In [0]:
# ==========================================
# INTAKE LAYER - CONFIG-DRIVEN DATA LOADING
# ==========================================
# This notebook loads all source files into the intake (bronze) layer
# using metadata from intake_config.py for maintainability and consistency

import sys
from pyspark.sql.functions import current_timestamp, lit, col

# Import intake configuration (note: import module name without .py extension)
sys.path.append('/Workspace/Repos/akshaymanikuttan05@gmail.com/AxioGo/axiogo_lakehouse/metadata')
from intake_config import INTAKE_TABLES, LOAD_ORDER

print("=" * 60)
print("INTAKE LAYER - AUTOMATED DATA LOADING")
print("=" * 60)
print(f"Total tables to load: {len(LOAD_ORDER)}\n")

# Track loading results
loaded_tables = []
failed_tables = []

# Loop through all tables in load order
for table_key in LOAD_ORDER:
    config = INTAKE_TABLES[table_key]
    
    try:
        # Check if table already exists (skip Excel files that are already loaded)
        table_exists = False
        try:
            existing_count = spark.table(config['table_name']).count()
            table_exists = True
            if config['file_format'] == 'excel':
                print(f"⏭️  Skipping: {table_key} (already exists with {existing_count:,} rows)\n")
                loaded_tables.append(table_key)
                continue
        except:
            table_exists = False
        
        print(f"📥 Loading: {table_key} ({config['batch']})...")
        
        # Handle different file formats
        if config['file_format'] == 'csv':
            # Load CSV files
            reader = spark.read
            for opt_key, opt_val in config['load_options'].items():
                reader = reader.option(opt_key, opt_val)
            df = reader.csv(config['source_path'])
            
        elif config['file_format'] == 'json':
            # Load JSON files
            reader = spark.read
            for opt_key, opt_val in config['load_options'].items():
                reader = reader.option(opt_key, opt_val)
            df = reader.json(config['source_path'])
            
        elif config['file_format'] == 'excel':
            # Excel files must be loaded manually (not supported in Serverless)
            # This code path should not be reached due to the skip logic above
            raise Exception(f"Excel format not supported in Serverless. Please load {table_key} manually.")
            
        elif config['file_format'] == 'pdf':
            # Load PDF files as binary
            reader = spark.read.format("binaryFile")
            for opt_key, opt_val in config['load_options'].items():
                reader = reader.option(opt_key, opt_val)
            df = reader.load(config['source_path'])
            
            # Add metadata columns for PDF files (using withColumns for better performance)
            df = df.withColumns({
                "source_file": col("_metadata.file_path"),
                "file_name": col("_metadata.file_name"),
                "file_size": col("_metadata.file_size"),
                "document_type": lit(config.get('document_type', 'Document')),
                "batch_id": lit(config['batch']),
                "ingestion_timestamp": current_timestamp()
            })
        
        # Write to Delta table and count rows (trigger action inside try block)
        df.write.format("delta").mode("overwrite").saveAsTable(config['table_name'])
        row_count = spark.table(config['table_name']).count()
        
        loaded_tables.append(table_key)
        print(f"   ✅ Success: {row_count:,} rows written to {config['table_name']}\n")
        
    except Exception as e:
        failed_tables.append((table_key, str(e)))
        print(f"   ❌ Failed: {str(e)}\n")

# Summary
print("=" * 60)
print("LOADING SUMMARY")
print("=" * 60)
print(f"✅ Successfully loaded: {len(loaded_tables)}/{len(LOAD_ORDER)} tables")
if failed_tables:
    print(f"❌ Failed: {len(failed_tables)} tables")
    for table, error in failed_tables:
        print(f"   - {table}: {error}")
else:
    print("🎉 All tables loaded successfully!")

